# 01. Six-Fold Leave-One-Dataset-Out (LODO) Split Generation

This notebook walks through the generation and mathematical verification of the 6-fold Leave-One-Dataset-Out (LODO) cross-domain generalization benchmark.


In [ ]:
import pandas as pd
from drdg.data.lodo import generate_lodo_folds, generate_and_save_lodo_splits
from drdg.data.splits import audit_patient_leakage

# Build synthetic 6-cohort demonstration dataset
cohorts = ["aptos", "ddr", "deepdrid", "idrid", "messidor2", "eyepacs"]
records = []
for c_idx, cohort in enumerate(cohorts):
    for p_idx in range(20):  # 20 patients per cohort
        pid = f"{cohort}::{p_idx:03d}"
        for eye in ["left", "right"]:
            records.append({
                "image_id": f"{cohort}_{p_idx}_{eye}",
                "patient_id": pid,
                "cohort": cohort,
                "dr_grade": (p_idx + c_idx) % 5,
                "eye": eye
            })
df_demo = pd.DataFrame(records)
print(f"Generated demo dataset: {len(df_demo)} images across {len(cohorts)} cohorts.")


## 1. Generate 6 LODO Folds with Strict Patient Isolation

In [ ]:
folds = generate_lodo_folds(df_demo, seed=42)
print(f"Successfully generated {len(folds)} verified LODO folds.")

for f in folds:
    f.verify_zero_leakage()
    print(f"Fold {f.fold_idx} | Target: {f.target_cohort:10s} | "
          f"Source Train: {len(f.source_train_df):3d} | "
          f"Source Val: {len(f.source_val_df):3d} | "
          f"Source Test: {len(f.source_test_df):3d} | "
          f"Target Test: {len(f.target_test_df):3d}")
